# Track B — Fine-tune Relation Classifier

Phân loại quan hệ **6 lớp** giữa hai atomic claim của hai reviewer:
`AGREEMENT · PARTIAL_AGREEMENT · COMPLEMENTARY · PARTIAL_CONTRADICTION · CONTRADICTION · UNRELATED`

- **Train** trên `phase2_trackb/processed/trackB_silver.jsonl` — nhãn LLM, tiếng Anh, review paper ICLR.
- **Chấm điểm cuối** trên `phase2_trackb/golden_set/gold_test.jsonl` — 129 cặp `HUMAN_VERIFIED`,
  tiếng Việt, phản biện đề tài đại học. Đây là con số duy nhất có nền người.

Contract nhãn: [`docs/RUBRIC_GOLD.md`](../docs/RUBRIC_GOLD.md) — rút ra từ chính tập gold.

> ⚠ **Notebook sẽ dừng ở mục 3** nếu `trackB_silver.jsonl` chưa được gán lại theo contract
> gold. Rubric cũ gán `COMPLEMENTARY` cho *"cùng aspect nhưng khác điểm cụ thể"* (494/496 cặp),
> còn gold gọi đúng tình huống đó là `UNRELATED`. Lớp này chiếm 45% dữ liệu train, nên train
> trước khi gán lại là dạy model trả lời ngược đáp án. Chạy `src/data/relabel_complementary.py`.

---

### Bốn lựa chọn thiết kế không phải mặc định

1. **Đối xứng theo cấu trúc.** Quan hệ không phụ thuộc claim nào đứng trước. Toàn bộ sự cố
   order-flip của pipeline ensemble (94% cặp phải đi debate chỉ vì có model đổi ý khi đảo A/B)
   đến từ chỗ này. Xử lý bằng **augment 2 chiều lúc train** + **cộng logit 2 chiều lúc suy luận**,
   và **đo trực tiếp flip-rate** để kiểm chứng chứ không chỉ hi vọng.
2. **Trọng số lớp.** CONTRADICTION chỉ ~2.6%. Không có trọng số thì model bỏ hẳn lớp này
   mà accuracy vẫn đẹp.
3. **Split đọc từ đĩa.** `src/train/split.py` chia theo nhóm paper + stratified theo nhãn +
   ghim few-shot vào train. Notebook **không** tự chia lại.
4. **Backbone đa ngữ.** `xlm-roberta-base` chứ không phải `roberta-base` — tập gold là tiếng
   Việt. Đây là đánh giá cross-lingual zero-shot: train EN, chấm VI.

### Thứ tự chạy
`Setup → Data → Split → Baselines → Train → Đánh giá → 5-fold CV → Learning curve → GOLD → Lưu checkpoint`

## 0 · Setup

In [ ]:
# Pin 4.44.2 đã fail build wheel cho tokenizers -> notebook âm thầm chạy bản Colab có sẵn,
# tức phiên bản KHÔNG tái lập được. Không pin nữa, nhưng IN RA bản thật để ghi vào manifest.
!pip install -q -U transformers scikit-learn
import torch, transformers, sklearn, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(f"torch {torch.__version__} | transformers {transformers.__version__} "
      f"| sklearn {sklearn.__version__} | {torch.cuda.get_device_name(0)}")

## 1 · Lấy dữ liệu

Repo public nên clone thẳng. **Nhớ push `trackB_silver.jsonl` lên GitHub trước khi chạy cell này.**

In [ ]:
import os, json
REPO = "https://github.com/navihat/build-phase2-finetune.git"
if not os.path.exists("build-phase2-finetune"):
    !git clone -q {REPO}
%cd build-phase2-finetune
!git pull -q
SILVER = "phase2_trackb/processed/trackB_silver.jsonl"

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

rows = read_jsonl(SILVER)
print(f"{len(rows)} cặp | {len({r['paper_id'].split('|')[0] for r in rows})} nhóm paper")

## 2 · Cấu hình

In [ ]:
from dataclasses import dataclass, replace

LABELS = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY",
          "PARTIAL_CONTRADICTION","CONTRADICTION","UNRELATED"]
L2I = {l:i for i,l in enumerate(LABELS)}
ALL_LABELS = LABELS

# Trục quan hệ: sai giữa hai nhãn KỀ NHAU nhẹ hơn nhiều so với sai giữa hai nhãn XA NHAU.
# UNRELATED nằm ngoài trục, chỉ kề COMPLEMENTARY.
AXIS = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY","PARTIAL_CONTRADICTION","CONTRADICTION"]

def axis_dist(a, b):
    """Khoảng cách trên trục quan hệ; UNRELATED cách COMPLEMENTARY 1 bước."""
    if a == b: return 0
    if "UNRELATED" in (a, b):
        other = b if a == "UNRELATED" else a
        return 1 if other == "COMPLEMENTARY" else 3
    return abs(AXIS.index(a) - AXIS.index(b))

@dataclass
class Cfg:
    # xlm-roberta-base chứ không phải roberta-base: tập gold (golden_set/gold_test.jsonl)
    # là 129/129 cặp TIẾNG VIỆT, còn silver là 1099/1099 tiếng Anh. roberta-base không đọc
    # được tiếng Việt. Đây là đánh giá cross-lingual zero-shot: train EN, chấm VI.
    model: str = "xlm-roberta-base"
    epochs: int = 6                # TRẦN. Epoch thật do val chọn (xem train_model)
    patience: int = 2
    batch: int = 16
    eval_batch: int = 64
    lr: float = 2e-5
    max_len: int = 160
    fp16: bool = True
    symmetric_aug: bool = True     # train: nhân đôi cặp theo 2 thứ tự
    symmetric_tta: bool = True     # infer: cộng logit 2 thứ tự
    seed: int = 20260828

    # Bỏ 110 cặp sinh bằng luật "ghép claim của hai paper khác nhau -> UNRELATED".
    # Sai cả hai mặt: (1) không học được — 91.8% số cặp đó không chung từ nội dung nào,
    # nhưng COMPLEMENTARY cùng paper cũng 83.9% như vậy, khác biệt duy nhất là paper_id mà
    # model không nhận; (2) sai định nghĩa — gold KHÔNG coi "khác paper" là UNRELATED, cả
    # 18 cặp UNRELATED của gold đều cùng cohort + cùng criterion, chỉ khác reviewer.
    # UNRELATED thật đến từ việc gán lại 496 cặp COMPLEMENTARY. Xem docs/RUBRIC_GOLD.md.
    drop_synthetic_unrelated: bool = True

cfg = Cfg()
print(cfg)
print(f"\n{len(LABELS)} lớp: {LABELS}")

## 3 · Đọc split đã chia sẵn

**Notebook không tự chia dữ liệu.** Split do [`src/train/split.py`](../src/train/split.py) sinh ra
và commit vào repo, nên notebook, script train và mọi lần chạy sau đều dùng đúng một bộ.

Bản trước của notebook có bản sao riêng của `assign_groups` và tự chia tại chỗ — đó là bản
**chưa stratified**, khiến test chỉ có 3 mẫu AGREEMENT / 2 mẫu CONTRADICTION và 5-fold lệch
13 lần ở CONTRADICTION. Mọi số đo ra từ bản đó không so sánh được với nhau.

Split hiện tại đảm bảo ba điều, kiểm bằng `assert` ở dưới:

1. **Nhóm theo paper** — không paper nào nằm ở hai phần (cặp chéo paper lấy paper trái làm khoá).
2. **Stratified theo nhãn** — lệch lớn nhất so với phân bố toàn cục là 2.2 điểm %.
3. **Few-shot ghim vào train** — 39 cặp `fewshot_human_pairs` chính là các ví dụ đã dùng để
   gán nhãn 1060 cặp còn lại, nên không được phép nằm ở phần chấm điểm.

Chi tiết: [`phase2_trackb/reports/split_report.md`](../phase2_trackb/reports/split_report.md).

In [ ]:
import random, collections
import numpy as np

SPLITS = "phase2_trackb/processed/splits"

def group_of(r): return r["paper_id"].split("|")[0]

def keep(part):
    return [r for r in part
            if not (cfg.drop_synthetic_unrelated and r["source"] == "synthetic_cross_paper")]

n_before   = len(rows)
rows       = keep(rows)
train_rows = keep(read_jsonl(f"{SPLITS}/trackB_train.jsonl"))
val_rows   = keep(read_jsonl(f"{SPLITS}/trackB_val.jsonl"))
test_rows  = keep(read_jsonl(f"{SPLITS}/trackB_test.jsonl"))
FOLDS      = json.load(open(f"{SPLITS}/folds.json", encoding="utf-8"))

# --- CHỐT CHẶN: silver phải được gán lại theo contract gold trước khi train ---
# Rubric cũ gán COMPLEMENTARY cho "cùng aspect nhưng KHÁC điểm cụ thể" (494/496 cặp),
# gold gọi đúng tình huống đó là UNRELATED. COMPLEMENTARY chiếm 45% dữ liệu train, nên
# train trước khi gán lại = dạy model trả lời ngược đáp án gold. Xem docs/RUBRIC_GOLD.md.
prov = collections.Counter(r.get("provenance") for r in rows)
n_unrel = sum(1 for r in rows if r["relation"] == "UNRELATED")
if "relabel_to_gold_contract_v1" not in prov or n_unrel < 30:
    raise SystemExit(
        f"\n  DỪNG: trackB_silver.jsonl chưa được gán lại theo contract gold.\n"
        f"  provenance hiện tại: {dict(prov)}\n"
        f"  UNRELATED còn {n_unrel} cặp (cần >=30 sau khi gán lại).\n\n"
        f"  Chạy trước:\n"
        f"    python src/data/relabel_complementary.py         # xuất 10 batch x 50 cặp\n"
        f"    <gán nhãn, gộp thành decisions.jsonl>\n"
        f"    python src/data/relabel_complementary.py --apply decisions.jsonl\n"
        f"    python src/train/split.py                        # chia lại sau khi silver đổi\n")

def subsample_groups(part, frac, seed=cfg.seed):
    """Lấy ~frac dữ liệu nhưng CẮT THEO NHÓM PAPER, không cắt giữa nhóm (cho learning curve)."""
    by = collections.defaultdict(list)
    for r in part: by[group_of(r)].append(r)
    g = sorted(by); random.Random(seed).shuffle(g)
    out, target = [], frac*len(part)
    for k in g:
        if len(out) >= target: break
        out.extend(by[k])
    return out

def summarize(name, part):
    d = collections.Counter(r["relation"] for r in part)
    print(f"{name:<7}{len(part):>5}  {len({group_of(r) for r in part}):>4} paper  " +
          "  ".join(f"{l[:4]}:{d.get(l,0)}" for l in LABELS))

for n_, p_ in [("train",train_rows),("val",val_rows),("test",test_rows)]: summarize(n_, p_)

gt,gv,gs = ({group_of(r) for r in p} for p in (train_rows,val_rows,test_rows))
assert not (gt&gv) and not (gt&gs) and not (gv&gs), "paper lọt sang phần khác"
assert not [r for r in val_rows + test_rows if r["source"] == "fewshot_human_pairs"], \
    "cặp few-shot lọt vào phần chấm điểm"
assert len(train_rows)+len(val_rows)+len(test_rows) == len(rows), \
    "split không phủ kín silver — chạy lại src/train/split.py sau khi silver đổi"
print("\n[OK] không paper nào nằm ở hai phần")
print("[OK] few-shot chỉ nằm trong train")
print("[OK] silver đã theo contract gold")
if cfg.drop_synthetic_unrelated:
    print(f"[!] đã bỏ synthetic_cross_paper: {n_before} -> {len(rows)} cặp")
print("[!] test chỉ ~110 cặp -> đọc số từ 5-fold CV; số CUỐI CÙNG lấy từ gold ở mục 10")

## 4 · Baseline — mốc để so sánh

Không có mốc thì macro-F1 = 0.45 là tốt hay tệ đều không biết.

- **majority**: luôn đoán COMPLEMENTARY (lớp đông nhất)
- **stance-rule**: dùng đúng tín hiệu đã dùng để đào cặp — stance đối nghịch → PARTIAL_CONTRADICTION

In [12]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_test = np.array([L2I[r["relation"]] for r in test_rows])

pred_majority = np.full(len(test_rows), L2I["COMPLEMENTARY"])

def stance_rule(r):
    s = {r["left"]["stance"], r["right"]["stance"]}
    if s == {"POSITIVE","NEGATIVE"}: return L2I["PARTIAL_CONTRADICTION"]
    if s == {"POSITIVE"}:            return L2I["PARTIAL_AGREEMENT"]
    return L2I["COMPLEMENTARY"]
pred_stance = np.array([stance_rule(r) for r in test_rows])

for name, p in [("majority", pred_majority), ("stance-rule", pred_stance)]:
    print(f"{name:<12} macro-F1={f1_score(y_test,p,average='macro',zero_division=0):.4f}  "
          f"micro-F1={f1_score(y_test,p,average='micro',zero_division=0):.4f}")

majority     macro-F1=0.1151  micro-F1=0.5273
stance-rule  macro-F1=0.2284  micro-F1=0.5182


## 5 · Model + vòng huấn luyện

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
device = torch.device("cuda")

class PairSet(Dataset):
    def __init__(self, rows, tok, max_len, symmetric_aug):
        self.ex = []
        for r in rows:
            y = L2I[r["relation"]]; a, b = r["left"]["text"], r["right"]["text"]
            self.ex.append((a,b,y))
            if symmetric_aug: self.ex.append((b,a,y))   # cùng nhãn, đảo thứ tự
        self.tok, self.max_len = tok, max_len
    def __len__(self): return len(self.ex)
    def __getitem__(self, i):
        a,b,y = self.ex[i]
        e = self.tok(a, b, truncation=True, max_length=self.max_len, padding=False)
        e["labels"] = y; return e

def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt"); out["labels"] = labels
        return out
    return collate

@torch.no_grad()
def predict_logits(model, tok, rows, symmetric_tta):
    """symmetric_tta=True: cộng logit của cả hai thứ tự (A,B) và (B,A)."""
    model.eval()
    orders = [(0,1)] + ([(1,0)] if symmetric_tta else [])
    total = None
    for lo, ro in orders:
        parts = []
        for i in range(0, len(rows), cfg.eval_batch):
            ch = rows[i:i+cfg.eval_batch]
            t = [(r["left"]["text"], r["right"]["text"]) for r in ch]
            enc = tok([x[lo] for x in t], [x[ro] for x in t], truncation=True,
                      max_length=cfg.max_len, padding=True, return_tensors="pt").to(device)
            parts.append(model(**enc).logits.float().cpu())
        lg = torch.cat(parts)
        total = lg if total is None else total + lg
    return total.numpy()

def train_model(train_rows, cfg, tag="", val_rows=None):
    """Có val_rows -> chọn epoch tốt nhất theo macro-F1 trên val, dừng sớm sau cfg.patience.

    Vì sao cần: bản trước train cứng 6 epoch và gộp val THẲNG vào train, nên không có gì
    canh overfit. Kết quả đo được: train loss xuống 0.24 trên ~1000 mẫu, ECE 0.338,
    confidence khi đúng 0.868 vs khi sai 0.840 -> không đặt nổi ngưỡng ABSTAIN.

    Không có val_rows -> train đủ cfg.epochs (nhánh này dùng cho 5-fold CV, vì trong CV
    không có tập val riêng; số epoch ở đó lấy từ epoch mà val đã chọn ở mục 6).
    """
    tok = AutoTokenizer.from_pretrained(cfg.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model, num_labels=len(LABELS)).to(device)
    dl = DataLoader(PairSet(train_rows, tok, cfg.max_len, cfg.symmetric_aug),
                    batch_size=cfg.batch, shuffle=True, collate_fn=make_collate(tok))
    # trọng số lớp inverse-frequency, tính trên chính tập train này
    cnt = collections.Counter(r["relation"] for r in train_rows)
    w = torch.tensor([len(train_rows)/(len(LABELS)*max(cnt.get(l,0),1)) for l in LABELS],
                     dtype=torch.float, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    steps = len(dl)*cfg.epochs
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(steps*0.1), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)

    y_val = np.array([L2I[r["relation"]] for r in val_rows]) if val_rows else None
    best_f1, best_ep, best_state = -1.0, cfg.epochs, None

    for ep in range(1, cfg.epochs+1):
        model.train(); run = 0.0
        for batch in dl:
            batch = {k:v.to(device) for k,v in batch.items()}
            labels = batch.pop("labels")
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.fp16):
                loss = loss_fn(model(**batch).logits.float(), labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        msg = f"  {tag}epoch {ep}/{cfg.epochs}  loss={run/len(dl):.4f}"
        if val_rows:
            vp = predict_logits(model, tok, val_rows, cfg.symmetric_tta).argmax(1)
            vf1 = f1_score(y_val, vp, average="macro", zero_division=0)
            msg += f"  val_macroF1={vf1:.4f}"
            if vf1 > best_f1:
                best_f1, best_ep = vf1, ep
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
                msg += "  *"
        print(msg)
        if val_rows and ep - best_ep >= cfg.patience:
            print(f"  {tag}dừng sớm: {cfg.patience} epoch liền không cải thiện val"); break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  {tag}-> giữ epoch {best_ep} (val macro-F1 {best_f1:.4f})")
    return model, tok, best_ep

print("sẵn sàng")

## 6 · Train (train → chọn epoch bằng val → chấm trên test)

Bản trước gọi `train_model(train_rows + val_rows, cfg)` — tức **gộp val thẳng vào train**,
val không làm nhiệm vụ của val. Cộng với 6 epoch cố định, model chạy tới train loss 0.24
trên ~1000 mẫu: thuộc lòng dữ liệu, ECE 0.338, confidence vô dụng.

Nay val giữ đúng vai: sau mỗi epoch chấm macro-F1 trên val, giữ trạng thái tốt nhất, dừng
sớm nếu `cfg.patience` epoch liền không cải thiện. Epoch được chọn ở đây sẽ dùng lại cho
5-fold CV — trong CV không có tập val riêng nên phải cố định số epoch từ trước.

In [ ]:
model, tok, BEST_EPOCH = train_model(train_rows, cfg, val_rows=val_rows)
logits_test = predict_logits(model, tok, test_rows, cfg.symmetric_tta)
y_pred = logits_test.argmax(1)
print(f"\nmacro-F1 (test) = {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"BEST_EPOCH = {BEST_EPOCH}  -> 5-fold CV ở mục 8 sẽ train đúng {BEST_EPOCH} epoch")

## 7 · Đánh giá

Bảy phép đo, mỗi phép trả lời một câu hỏi khác nhau. Đừng chỉ đọc accuracy.

### 7.1 · Per-class P/R/F1 — *lớp nào model bỏ rơi?*

Với phân bố lệch 45% / 2.6%, **accuracy vô dụng**: đoán COMPLEMENTARY hết vẫn được ~45%.
Macro-F1 là số chính, vì nó cho mọi lớp trọng số bằng nhau.

In [15]:
present = sorted(set(y_test) | set(y_pred))
print(classification_report(y_test, y_pred, labels=present,
      target_names=[LABELS[i] for i in present], digits=3, zero_division=0))
print(f"macro-F1={f1_score(y_test,y_pred,average='macro',zero_division=0):.4f}   "
      f"micro-F1={f1_score(y_test,y_pred,average='micro',zero_division=0):.4f}")

                       precision    recall  f1-score   support

            AGREEMENT      0.000     0.000     0.000         3
    PARTIAL_AGREEMENT      0.375     0.500     0.429        12
        COMPLEMENTARY      0.667     0.621     0.643        58
PARTIAL_CONTRADICTION      0.467     0.560     0.509        25
        CONTRADICTION      0.000     0.000     0.000         2
            UNRELATED      0.143     0.100     0.118        10

             accuracy                          0.518       110
            macro avg      0.275     0.297     0.283       110
         weighted avg      0.511     0.518     0.512       110

macro-F1=0.2830   micro-F1=0.5182


### 7.2 · Ma trận nhầm lẫn — *sai theo MẪU nào?*

Tìm **mẫu lỗi hệ thống**, không phải tỉ lệ %. Một ranh giới lệch đều một hướng
(ví dụ PARTIAL_CONTRADICTION luôn bị đoán thành CONTRADICTION) là dấu hiệu rubric
mờ ở đúng ranh giới đó, không phải model kém.

In [ ]:
K = len(LABELS)
cm = confusion_matrix(y_test, y_pred, labels=list(range(K)))
print(" "*26 + "".join(f"{l[:6]:>8}" for l in LABELS) + "     n")
for i,l in enumerate(LABELS):
    print(f"{l:<26}" + "".join(f"{v:>8}" for v in cm[i]) + f"  {cm[i].sum():>5}")

print("\nCác ô nhầm nhiều nhất:")
err = [(cm[i][j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K) if i!=j and cm[i][j]>0]
for n,t,p in sorted(err, reverse=True)[:8]:
    print(f"  {n:>3}x  {t}  ->  {p}   (cách {axis_dist(t,p)} bước trên trục)")

### 7.3 · Độ chính xác có dung sai theo trục — *sai NẶNG hay sai NHẸ?*

6 nhãn nằm trên một trục liên tục. Nhầm `PARTIAL_CONTRADICTION` ↔ `CONTRADICTION`
(kề nhau) nhẹ hơn hẳn nhầm `AGREEMENT` ↔ `CONTRADICTION` (cách 4 bước).
Accuracy thường coi hai lỗi này như nhau — đây là chỗ nó che mất sự thật.

In [17]:
d = np.array([axis_dist(LABELS[t], LABELS[p]) for t,p in zip(y_test, y_pred)])
print(f"đúng tuyệt đối (d=0)      : {(d==0).mean():.3f}")
print(f"đúng hoặc lệch 1 bước     : {(d<=1).mean():.3f}   <- chỉ số dễ đọc nhất cho ứng dụng")
print(f"lệch >=2 bước (sai nặng)  : {(d>=2).mean():.3f}")
print(f"khoảng cách trung bình    : {d.mean():.3f} bước")
print("\nphân bố khoảng cách lỗi:", dict(collections.Counter(d.tolist())))

đúng tuyệt đối (d=0)      : 0.518
đúng hoặc lệch 1 bước     : 0.873   <- chỉ số dễ đọc nhất cho ứng dụng
lệch >=2 bước (sai nặng)  : 0.127
khoảng cách trung bình    : 0.682 bước

phân bố khoảng cách lỗi: {0: 57, 1: 39, 2: 7, 3: 6, 4: 1}


### 7.4 · Quadratic Weighted Kappa — *hơn đoán mò bao nhiêu, có tính thứ tự?*

QWK phạt lỗi theo **bình phương khoảng cách** và hiệu chỉnh theo mức đồng thuận ngẫu nhiên.
Đây là thước đo phù hợp nhất cho nhãn có thứ tự. Bỏ UNRELATED vì nó nằm ngoài trục.

Đọc: <0.2 kém · 0.4-0.6 khá · >0.6 tốt · >0.8 rất tốt

In [18]:
from sklearn.metrics import cohen_kappa_score
ax_idx = {l:i for i,l in enumerate(AXIS)}
mask = np.array([LABELS[t] in ax_idx and LABELS[p] in ax_idx for t,p in zip(y_test,y_pred)])
if mask.sum() > 1:
    yt = [ax_idx[LABELS[t]] for t,m in zip(y_test,mask) if m]
    yp = [ax_idx[LABELS[p]] for p,m in zip(y_pred,mask) if m]
    print(f"QWK trên trục (n={mask.sum()}): {cohen_kappa_score(yt,yp,weights='quadratic'):.4f}")
print(f"Kappa thường (cả 6 lớp)  : {cohen_kappa_score(y_test,y_pred):.4f}")

QWK trên trục (n=94): 0.2400
Kappa thường (cả 6 lớp)  : 0.2665


### 7.5 · Flip-rate — *model có ĐỐI XỨNG thật không?*

**Đây là phép đo quan trọng nhất của dự án này.** Pipeline ensemble sụp đổ vì
Qwen lật nhãn 50% và Gemma 44% khi đảo A/B, khiến 94% cặp phải đi debate.

Đo trên logit THÔ (tắt TTA) để biết model tự nó đã đối xứng chưa. TTA làm flip-rate = 0
theo định nghĩa, nên nếu chỉ đo có TTA thì không phát hiện được vấn đề.

In [19]:
raw_fwd = predict_logits(model, tok, test_rows, symmetric_tta=False)
swapped = [{**r, "left": r["right"], "right": r["left"]} for r in test_rows]
raw_rev = predict_logits(model, tok, swapped, symmetric_tta=False)
f_, r_ = raw_fwd.argmax(1), raw_rev.argmax(1)
flip = (f_ != r_)
print(f"flip-rate (thô, không TTA) : {flip.mean():.3f}   ({flip.sum()}/{len(flip)} cặp)")
if flip.sum():
    dd = [axis_dist(LABELS[a], LABELS[b]) for a,b in zip(f_[flip], r_[flip])]
    print(f"  trong đó lệch 1 bước     : {sum(1 for x in dd if x==1)}/{len(dd)}")
print("\nĐối chiếu pipeline ensemble cũ: Qwen 50%, Gemma 44%, SeaLLM 18%.")
print("Dưới ~10% là đã khắc phục được vấn đề đã làm hỏng Track B.")

flip-rate (thô, không TTA) : 0.155   (17/110 cặp)
  trong đó lệch 1 bước     : 16/17

Đối chiếu pipeline ensemble cũ: Qwen 50%, Gemma 44%, SeaLLM 18%.
Dưới ~10% là đã khắc phục được vấn đề đã làm hỏng Track B.


### 7.6 · Tách theo NGUỒN dữ liệu — *con số nào là thật?*

Dữ liệu đến từ nhiều nguồn có độ tin cậy khác nhau. Bảng này bắt loại tự lừa kiểu
"model giỏi ở nguồn dễ, kém ở nguồn thật, nhưng con số tổng vẫn đẹp".

Lần chạy trước đã bắt được đúng một ca như vậy, nhưng theo chiều ngược lại với dự đoán:
`synthetic_cross_paper` cho **acc 0.100 / macro-F1 0.045** — tệ nhất trong mọi nguồn, dù nó
là nguồn *dễ đoán nhất* trên lý thuyết. Đó là dấu hiệu dẫn tới việc bỏ hẳn lớp UNRELATED
(xem `cfg.drop_unrelated` và `docs/TRAIN.md` muc 9).

Với `drop_unrelated=True`, nguồn đó không còn xuất hiện trong bảng dưới.

In [20]:
by_src = collections.defaultdict(list)
for i,r in enumerate(test_rows): by_src[r["source"]].append(i)
print(f"{'nguồn':<28}{'n':>5}{'acc':>8}{'macroF1':>9}")
for s, idx in sorted(by_src.items()):
    yt_, yp_ = y_test[idx], y_pred[idx]
    print(f"{s:<28}{len(idx):>5}{(yt_==yp_).mean():>8.3f}"
          f"{f1_score(yt_,yp_,average='macro',zero_division=0):>9.3f}")

nguồn                           n     acc  macroF1
full_batch00_manual            20   0.550    0.313
full_batch01_manual            18   0.278    0.172
full_batch02_manual            18   0.722    0.446
full_batch03_manual            16   0.750    0.407
mined_stance_opposition        28   0.536    0.328
synthetic_cross_paper          10   0.100    0.045


### 7.7 · Hiệu chuẩn — *có thể đặt ngưỡng ABSTAIN không?*

Nếu model tự tin sai nhiều thì không dùng được confidence để lọc. Bảng này cho biết
nên cắt ngưỡng ở đâu nếu muốn đánh đổi coverage lấy độ chính xác.

In [21]:
probs = torch.softmax(torch.tensor(logits_test), dim=1).numpy()
conf = probs.max(1); correct = (y_pred == y_test)
print(f"confidence trung bình: đúng={conf[correct].mean():.3f}  sai={conf[~correct].mean():.3f}")
print(f"\n{'ngưỡng':>8}{'coverage':>11}{'acc trên phần giữ lại':>24}")
for t in [0.0,0.5,0.6,0.7,0.8,0.9]:
    m = conf >= t
    if m.sum():
        print(f"{t:>8.1f}{m.mean():>11.3f}{correct[m].mean():>24.3f}")

# ECE 10 bin
bins = np.linspace(0,1,11); ece = 0.0
for lo,hi in zip(bins[:-1],bins[1:]):
    m = (conf>lo)&(conf<=hi)
    if m.sum(): ece += m.mean()*abs(correct[m].mean()-conf[m].mean())
print(f"\nECE = {ece:.4f}   (<0.05 hiệu chuẩn tốt, >0.15 quá tự tin)")

confidence trung bình: đúng=0.868  sai=0.840

  ngưỡng   coverage   acc trên phần giữ lại
     0.0      1.000                   0.518
     0.5      0.982                   0.519
     0.6      0.845                   0.538
     0.7      0.791                   0.552
     0.8      0.682                   0.547
     0.9      0.582                   0.547

ECE = 0.3379   (<0.05 hiệu chuẩn tốt, >0.15 quá tự tin)


## 8 · 5-fold CV — con số đáng tin

Test chỉ ~110 cặp, CONTRADICTION đúng 2 mẫu → F1 lớp đó trên test là số ngẫu nhiên.
CV cho **mỗi cặp được dự đoán đúng một lần** bởi một model chưa từng thấy nó, nên mọi lớp
đều đủ mẫu (CONTRADICTION: 22 thay vì 2).

Fold đọc từ `folds.json`, **không chia lại tại chỗ**. Bộ fold này đã stratified: CONTRADICTION
4–5 mỗi fold, trước khi sửa là 1–13. Cặp `fold = -1` là few-shot bị ghim, luôn nằm trong train
của mọi vòng và không bao giờ bị chấm điểm.

Mỗi vòng dựng **model mới hoàn toàn** từ checkpoint gốc — không có trọng số nào đi từ vòng
trước sang vòng sau. CV không có tập val riêng nên train cứng `BEST_EPOCH` epoch, tức số
epoch mà val đã chọn ở mục 6.

**Đây mới là con số để báo cáo.** Chạy ~5× lâu hơn.

In [ ]:
N_FOLDS = FOLDS["n_folds"]
fold_of = FOLDS["fold_of_pair_id"]
# BEST_EPOCH do val chọn ở mục 6; nếu chưa chạy mục 6 thì lấy 3 (thay vì 6 như bản cũ)
cv_cfg = replace(cfg, epochs=globals().get("BEST_EPOCH", 3))
print(f"CV train {cv_cfg.epochs} epoch/fold | ghim vào train mọi vòng: "
      f"{sum(1 for r in rows if fold_of[r['pair_id']] == -1)} cặp\n")

all_t, all_p, fold_f1 = [], [], []
for k in range(N_FOLDS):
    tr = [r for r in rows if fold_of[r["pair_id"]] != k]     # -1 rơi vào đây ở MỌI vòng
    te = [r for r in rows if fold_of[r["pair_id"]] == k]
    nc = collections.Counter(r["relation"] for r in te)
    print(f"\n--- fold {k}: train={len(tr)} test={len(te)}  "
          + " ".join(f"{l[:4]}:{nc.get(l,0)}" for l in LABELS) + " ---")
    m_, t_, _ = train_model(tr, cv_cfg, tag=f"[f{k}] ")
    yp_ = predict_logits(m_, t_, te, cv_cfg.symmetric_tta).argmax(1)
    yt_ = np.array([L2I[r["relation"]] for r in te])
    s = f1_score(yt_, yp_, average="macro", zero_division=0); fold_f1.append(s)
    print(f"  fold {k} macro-F1 = {s:.4f}")
    all_t.append(yt_); all_p.append(yp_)
    del m_; torch.cuda.empty_cache()

cv_t, cv_p = np.concatenate(all_t), np.concatenate(all_p)
print("\n" + "="*62)
print(f"macro-F1 từng fold: {[f'{s:.3f}' for s in fold_f1]}")
print(f"trung bình = {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print("="*62)
print(classification_report(cv_t, cv_p, target_names=LABELS, digits=3, zero_division=0))
dcv = np.array([axis_dist(LABELS[t],LABELS[p]) for t,p in zip(cv_t,cv_p)])
print(f"đúng hoặc lệch 1 bước: {(dcv<=1).mean():.3f}   sai nặng (>=2): {(dcv>=2).mean():.3f}")

# Mốc so sánh, tính trên ĐÚNG tập vừa chấm -> so được trực tiếp với con số trên
maj = np.full(len(cv_t), L2I["COMPLEMENTARY"])
print(f"\nmajority baseline trên cùng tập: macro-F1 = "
      f"{f1_score(cv_t, maj, average='macro', zero_division=0):.4f}")

## 9 · Learning curve — *thêm dữ liệu có đáng không?*

Trả lời bằng số cho câu "1099 cặp đã đủ chưa": train trên 25/50/75/100% rồi nhìn độ dốc.
Còn dốc → gán thêm nhãn sẽ có lời. Đã phẳng → tiền nên đổ vào chỗ khác (model lớn hơn,
sửa rubric, cân bằng lớp).

In [ ]:
curve = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    sub = subsample_groups(train_rows, frac) if frac < 1 else train_rows
    m_, t_, _ = train_model(sub, cfg, tag=f"[{int(frac*100)}%] ", val_rows=val_rows)
    yp_ = predict_logits(m_, t_, test_rows, cfg.symmetric_tta).argmax(1)
    s = f1_score(y_test, yp_, average="macro", zero_division=0)
    curve.append((len(sub), s)); print(f"  n={len(sub):<5} macro-F1={s:.4f}")
    del m_; torch.cuda.empty_cache()

print("\n n_train   macro-F1   Δ so với mức trước")
prev = None
for n_,s_ in curve:
    print(f"{n_:>8}{s_:>11.4f}" + (f"{s_-prev:>+12.4f}" if prev is not None else ""))
    prev = s_
print("\nΔ cuối còn lớn -> gán thêm nhãn còn lời. Δ ~0 -> đã bão hoà.")
print("⚠ Mỗi mức chỉ chạy MỘT seed và chấm trên test ~110 cặp. Lần chạy trước ra dãy")
print("  0.226 / 0.169 / 0.220 / 0.268 — mức 50% thấp hơn mức 25%, tức nhiễu giữa các lần")
print("  chạy lớn hơn cả hiệu ứng của dữ liệu. Muốn dùng đường này để RA QUYẾT ĐỊNH thì")
print("  phải chạy >=3 seed mỗi mức và chấm bằng CV, không phải test. Xem docs/TRAIN.md muc 9.")

## 10 · Đánh giá trên GOLD — con số duy nhất có nền người

`golden_set/gold_test.jsonl`: 129 cặp, `HUMAN_VERIFIED`, annotator `NTH`, phân bố **cân đều
cả 6 lớp** (COMP 29 · PART_CONTRA 24 · PART_AGREE 24 · AGREE 20 · UNREL 18 · CONTRA 14).

Mọi con số ở các mục trên đo **độ khớp với nhãn Claude**. Đây là chỗ duy nhất đo **độ khớp
với người**. Nếu hai con số lệch xa nhau thì cái đáng tin là con số ở đây.

Ba khoảng cách phải đọc kèm, nếu không sẽ quy sai nguyên nhân:

| | train (silver) | eval (gold) |
|---|---|---|
| Ngôn ngữ | 1099/1099 tiếng Anh | **129/129 tiếng Việt** |
| Miền | review paper ICLR về ML | phản biện đề tài đại học VN |
| Nhãn | rubric v1 | contract gold ([`RUBRIC_GOLD.md`](../docs/RUBRIC_GOLD.md)) |

Nên đây là **cross-lingual zero-shot + đổi miền**. `xlm-roberta-base` xử lý khoảng cách thứ
nhất; khoảng cách thứ ba đã xử lý bằng việc gán lại silver; khoảng cách thứ hai thì chưa —
và bảng theo cohort ở dưới cho biết nó tốn bao nhiêu.

In [ ]:
GOLD = "phase2_trackb/golden_set/gold_test.jsonl"
gold_raw = read_jsonl(GOLD)

# predict_logits chỉ cần left.text / right.text — nắn về đúng dạng đó
gold_rows = [{"pair_id": r["example_id"], "paper_id": r["cohort_id"],
              "aspect": r["criterion_id"], "source": "gold",
              "left": {"text": r["left"]["text"]}, "right": {"text": r["right"]["text"]},
              "relation": r["expected_relation"]} for r in gold_raw]
assert all(r["relation"] in L2I for r in gold_rows), \
    "gold có nhãn ngoài LABELS — kiểm tra lại cfg / docs/RUBRIC_GOLD.md"

y_gold = np.array([L2I[r["relation"]] for r in gold_rows])
logits_gold = predict_logits(model, tok, gold_rows, cfg.symmetric_tta)
p_gold = logits_gold.argmax(1)

print("="*64)
print(f"GOLD (n={len(gold_rows)}, tiếng Việt, HUMAN_VERIFIED)")
print("="*64)
print(classification_report(y_gold, p_gold, labels=list(range(len(LABELS))),
                            target_names=LABELS, digits=3, zero_division=0))
mf1 = f1_score(y_gold, p_gold, average="macro", zero_division=0)
print(f"macro-F1 (gold) = {mf1:.4f}    |    macro-F1 (silver CV) = so ở mục 8")

# mốc trên chính tập gold, để biết 'hơn đoán bừa' bao nhiêu
maj_g = np.full(len(y_gold), L2I["COMPLEMENTARY"])
print(f"majority baseline trên gold: {f1_score(y_gold, maj_g, average='macro', zero_division=0):.4f}")

# sai nặng hay nhẹ
dg = np.array([axis_dist(LABELS[t], LABELS[p]) for t,p in zip(y_gold, p_gold)])
print(f"\nđúng tuyệt đối {(dg==0).mean():.3f}  |  đúng-hoặc-lệch-1-bước {(dg<=1).mean():.3f}"
      f"  |  sai nặng (>=2) {(dg>=2).mean():.3f}")

# ma trận nhầm lẫn — chỗ đọc ra rubric còn lệch ở đâu
K = len(LABELS)
cmg = confusion_matrix(y_gold, p_gold, labels=list(range(K)))
print("\n" + " "*26 + "".join(f"{l[:6]:>8}" for l in LABELS) + "     n")
for i,l in enumerate(LABELS):
    print(f"{l:<26}" + "".join(f"{v:>8}" for v in cmg[i]) + f"  {cmg[i].sum():>5}")
print("\nÔ nhầm nhiều nhất:")
eg = [(cmg[i][j], LABELS[i], LABELS[j]) for i in range(K) for j in range(K)
      if i!=j and cmg[i][j]>0]
for n_,t_,p_ in sorted(eg, reverse=True)[:6]:
    print(f"  {n_:>3}x  {t_} -> {p_}   (cách {axis_dist(t_,p_)} bước)")
print("  ^ nếu COMPLEMENTARY <-> UNRELATED vẫn dẫn đầu thì việc gán lại chưa tới nơi")

# tách theo cohort: đo khoảng cách MIỀN, không phải khoảng cách nhãn
print(f"\n{'cohort':<34}{'n':>4}{'acc':>8}{'macroF1':>9}")
by_c = collections.defaultdict(list)
for i,r in enumerate(gold_rows): by_c[r["paper_id"]].append(i)
for c, idx in sorted(by_c.items()):
    yt_, yp_ = y_gold[idx], p_gold[idx]
    print(f"{c:<34}{len(idx):>4}{(yt_==yp_).mean():>8.3f}"
          f"{f1_score(yt_,yp_,average='macro',zero_division=0):>9.3f}")

# flip-rate trên gold, logit thô — model có đối xứng khi đổi sang tiếng Việt không
raw_f = predict_logits(model, tok, gold_rows, symmetric_tta=False)
swap_g = [{**r, "left": r["right"], "right": r["left"]} for r in gold_rows]
raw_r = predict_logits(model, tok, swap_g, symmetric_tta=False)
fl = (raw_f.argmax(1) != raw_r.argmax(1))
print(f"\nflip-rate trên gold (thô, không TTA): {fl.mean():.3f}  ({fl.sum()}/{len(fl)})")
print("  đối chiếu: flip-rate trên test tiếng Anh ở mục 7.5")

## 11 · Lưu checkpoint về Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/phase2_trackb/relation_classifier_v1"
model.save_pretrained(OUT); tok.save_pretrained(OUT)
with open(OUT + "/labels.json", "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS,
               "dropped": [l for l in ALL_LABELS if l not in L2I],
               "symmetric_tta": cfg.symmetric_tta,
               "best_epoch": BEST_EPOCH,
               "split": "phase2_trackb/processed/splits (stratified, few-shot ghim vào train)",
               "transformers": transformers.__version__}, f, ensure_ascii=False, indent=1)
print("đã lưu ->", OUT)
print("\nInference tại máy (GTX 1650 4GB thừa sức):")
print("  m = AutoModelForSequenceClassification.from_pretrained(OUT)")
print("  # nhớ cộng logit cả hai thứ tự (A,B) và (B,A) như lúc đánh giá")
if cfg.drop_unrelated:
    print("\n⚠ Checkpoint này KHÔNG có lớp UNRELATED. Hệ thống gọi nó phải tự đảm bảo")
    print("  hai claim đưa vào là của CÙNG MỘT PAPER — đó vốn là điều kiện của bài toán.")